<a href="https://colab.research.google.com/github/shakirali/nebius/blob/main/homework_lora.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework: LoRA from Scratch on GPT-2

In this assignment you will:

1. Implement **LoRA** (Low-Rank Adaptation) from scratch as a thin wrapper around `nn.Linear`.
2. Inject it into a frozen pre-trained **GPT-2 (small, 124M)** and verify only the LoRA params are trainable.
3. Fine-tune on **TinyShakespeare** so the model starts producing pseudo-Elizabethan text.
4. Save / load just the adapter (a few MB instead of 500MB).
5. Re-do the same fine-tune with the `peft` library and confirm the results match.
6. **Bonus:** swap the dataset for Rick & Morty dialogue, Hermione Granger lines, or Yoda quotes and watch the style transfer.

The whole thing fits on a free Colab T4 in well under an hour.

> **Submission:** run all cells, keep the printed sample outputs, and save the notebook. Written answers go in the `# YOUR ANSWER:` markdown cells.


## 0 · LoRA in one minute

For any frozen linear layer $W_0 \in \mathbb{R}^{d_{\text{out}} \times d_{\text{in}}}$, LoRA adds a learned low-rank update:

$$
h = W_0 x + \Delta W \cdot x, \qquad \Delta W = \frac{\alpha}{r}\, B A
$$

where $A \in \mathbb{R}^{r \times d_{\text{in}}}$ and $B \in \mathbb{R}^{d_{\text{out}} \times r}$ with $r \ll \min(d_{\text{in}}, d_{\text{out}})$.

Key facts:

- $A$ is initialized with Kaiming uniform, $B$ with **zeros** ⇒ $\Delta W = 0$ at init, so the wrapped model behaves identically to the original on step 0.
- Only $A$ and $B$ are trained ⇒ the trainable param count drops by ~3 orders of magnitude.
- At inference you can either (a) keep $A, B$ separate, or (b) merge: $W \leftarrow W_0 + \frac{\alpha}{r} B A$ — same FLOPs as the original.
- $\alpha$ is a scaling hyperparameter; effective learning rate of the update scales with $\alpha/r$.

Reference: Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models*, 2021 — https://arxiv.org/abs/2106.09685


## 1 · Setup

If you're on Colab, run the install cell. Local installs may already have these.


In [ ]:
# Colab install. Skip if you already have these locally.
!pip -q install "transformers>=4.40" "datasets>=2.18" "peft>=0.10" "accelerate>=0.27"

In [ ]:
import math
import os
import random
from dataclasses import dataclass
from typing import Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

import transformers
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device, "| transformers:", transformers.__version__)

## 2 · Inspect GPT-2

We'll use **gpt2** (124M params). Let's load it and see what kinds of layers live inside.

> **Heads up:** HuggingFace's GPT-2 uses `transformers.pytorch_utils.Conv1D` (a quirky historical choice — it's just a linear layer with weight transposed) for `c_attn` and `c_proj`, **not** `nn.Linear`. That makes writing a generic LoRA wrapper annoying. We'll convert these to plain `nn.Linear` first so your LoRA code stays clean.


In [ ]:
MODEL_NAME = "gpt2"

tokenizer = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

n_total = sum(p.numel() for p in model.parameters())
print(f"Total params: {n_total/1e6:.1f}M")

# Print one transformer block to see the layer types
print(model.transformer.h[0])

In [ ]:
# Helper: convert all transformers Conv1D modules to nn.Linear (functionally identical).
# This is given to you. Read it — but you don't need to modify it.
from transformers.pytorch_utils import Conv1D


def conv1d_to_linear(conv: Conv1D) -> nn.Linear:
    # Conv1D stores weight as (in_features, out_features); nn.Linear stores (out_features, in_features).
    in_features, out_features = conv.weight.shape
    linear = nn.Linear(in_features, out_features, bias=conv.bias is not None)
    with torch.no_grad():
        linear.weight.copy_(conv.weight.T)
        if conv.bias is not None:
            linear.bias.copy_(conv.bias)
    return linear


def replace_conv1d_with_linear(module: nn.Module) -> None:
    for name, child in list(module.named_children()):
        if isinstance(child, Conv1D):
            setattr(module, name, conv1d_to_linear(child).to(next(module.parameters()).device))
        else:
            replace_conv1d_with_linear(child)


replace_conv1d_with_linear(model)

# Sanity-check: the model still produces the same logits as before within fp tolerance.
with torch.no_grad():
    ids = tokenizer("The quick brown fox", return_tensors="pt").input_ids.to(device)
    logits = model(ids).logits
print("After conversion, logits shape:", tuple(logits.shape))
print(model.transformer.h[0].attn)  # c_attn / c_proj are now nn.Linear

## 3 · Baseline generation (before fine-tuning)

So we have a "before" reference for the style transfer. Save these outputs — you'll diff them against the post-LoRA samples later.


In [ ]:
@torch.no_grad()
def generate(prompt: str, max_new_tokens: int = 80, temperature: float = 0.9, top_p: float = 0.95, seed: int = 0) -> str:
    torch.manual_seed(seed)
    model.eval()
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    out = model.generate(
        ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True)


PROMPTS = [
    "ROMEO:",
    "To be, or not to be,",
    "Once upon a time in fair Verona,",
]

print("=== BASELINE (no fine-tuning) ===")
for p in PROMPTS:
    print("-" * 60)
    print(generate(p, seed=SEED))

## 4 · TODO: Implement `LoRALinear` (10 points)

Wrap an existing `nn.Linear` so the forward pass becomes:

$$
y = W_0 x + b + \frac{\alpha}{r} B A \cdot \text{dropout}(x)
$$

Constraints:

- The base layer's weight and bias must be **frozen** (`requires_grad = False`).
- $A$: shape `(r, in_features)`, initialised with `nn.init.kaiming_uniform_(a=math.sqrt(5))`.
- $B$: shape `(out_features, r)`, initialised with **zeros**. (Why zeros? See the background section.)
- Optional dropout on the input *before* it hits the LoRA branch.
- `merged_weight()` returns $W_0 + \frac{\alpha}{r} B A$ — useful for checking equivalence and for inference-time merging.

Fill in the `# TODO` lines.


In [ ]:
class LoRALinear(nn.Module):
    """Frozen linear layer + trainable low-rank residual."""

    def __init__(self, base: nn.Linear, r: int = 8, alpha: int = 16, dropout: float = 0.0):
        super().__init__()
        assert isinstance(base, nn.Linear), "LoRALinear only wraps nn.Linear in this homework."
        self.base = base
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_features = base.in_features
        out_features = base.out_features

        # TODO 4.1: freeze base weight & bias.

        # TODO 4.2: create the low-rank parameters.
        #   self.lora_A: Parameter of shape (r, in_features)
        #   self.lora_B: Parameter of shape (out_features, r)
        # Initialize A with kaiming_uniform_(a=math.sqrt(5)) and B with zeros.
        # IMPORTANT: allocate them on `base.weight.device` with `base.weight.dtype`,
        # otherwise on a GPU run you'll hit "Expected all tensors to be on the same device".

        # TODO 4.3: dropout on input. Use nn.Dropout(dropout) if dropout > 0 else nn.Identity().
        self.lora_dropout = ...

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # TODO 4.4: return base(x) + scaling * dropout(x) @ A.T @ B.T
        ...

    @torch.no_grad()
    def merged_weight(self) -> torch.Tensor:
        # TODO 4.5: return W0 + scaling * B @ A    (shape: out_features x in_features)
        ...

In [ ]:
# Sanity check: at init, LoRALinear must be functionally identical to the base layer.
torch.manual_seed(0)
base = nn.Linear(64, 32).to(device)
lora = LoRALinear(base, r=4, alpha=8)  # NOTE: no trailing .to(device) -- LoRALinear should already place params on base's device.

assert lora.lora_A.device == base.weight.device, (
    f"lora_A on {lora.lora_A.device} but base on {base.weight.device}. "
    "Allocate lora_A/lora_B on base.weight.device inside __init__."
)
assert lora.lora_B.device == base.weight.device, "lora_B on wrong device, see hint in TODO 4.2."

x = torch.randn(2, 5, 64, device=device)
y_base = base(x)
y_lora = lora(x)

assert torch.allclose(y_base, y_lora, atol=1e-6), "LoRA forward differs from base at init — B should be zeros!"
print("OK: LoRALinear matches base at init.")

# After perturbing B, outputs should differ.
with torch.no_grad():
    lora.lora_B.add_(torch.randn_like(lora.lora_B))
assert not torch.allclose(base(x), lora(x), atol=1e-6)
print("OK: LoRALinear differs from base after perturbing B.")

# Merged weight equivalence.
with torch.no_grad():
    merged = lora.merged_weight()
    y_merged = F.linear(x, merged, base.bias)
    assert torch.allclose(y_merged, lora(x), atol=1e-5), "merged_weight() inconsistent with forward()."
print("OK: merged_weight() matches forward().")

## 5 · TODO: Inject LoRA into GPT-2 and freeze the rest (5 points)

You'll write `inject_lora` that walks the model, finds every `nn.Linear` whose attribute name matches `target_names`, and replaces it with a `LoRALinear` wrapping it.

We'll target `("c_attn", "c_proj")`:

- `c_attn` — the fused QKV projection (one per block, in `attn`).
- `c_proj` — *appears in two places* per block: the attention output projection (`attn.c_proj`) **and** the MLP down-projection (`mlp.c_proj`). Matching by attribute name will hit both — that's fine and is consistent with what `peft` does by default (it suffix-matches the same names). It's a useful gotcha to be aware of.

So you should end up with **36 wrappings** = 12 blocks × (1 `c_attn` + 1 `attn.c_proj` + 1 `mlp.c_proj`).

After injection, **only `lora_A` and `lora_B` should have `requires_grad=True`.**


In [ ]:
def inject_lora(
    module: nn.Module,
    target_names: tuple[str, ...] = ("c_attn", "c_proj"),
    r: int = 8,
    alpha: int = 16,
    dropout: float = 0.05,
) -> int:
    """Recursively swap nn.Linear children whose attribute name is in target_names.

    Returns the number of modules wrapped.
    """
    n_wrapped = 0
    # TODO 5.1: walk module.named_children(), replace matching nn.Linear with LoRALinear,
    # recurse into non-matching children. Increment n_wrapped for each replacement.
    ...
    return n_wrapped


def freeze_non_lora(model: nn.Module) -> None:
    # TODO 5.2: set requires_grad=False on every parameter whose name does not contain "lora_".
    ...


def count_params(model: nn.Module) -> tuple[int, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total


def assert_same_device(model: nn.Module) -> None:
    """Sanity check: every parameter lives on the same device. Catches a common LoRA bug."""
    expected = next(model.parameters()).device
    for n, p in model.named_parameters():
        assert p.device == expected, f"param {n} on {p.device}, expected {expected}"

In [ ]:
# Reload a fresh GPT-2 (we'll keep the converted Linear version so LoRALinear can wrap it).
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
replace_conv1d_with_linear(model)

n_wrapped = inject_lora(model, target_names=("c_attn", "c_proj"), r=8, alpha=16, dropout=0.05)
freeze_non_lora(model)

trainable, total = count_params(model)
print(f"Wrapped {n_wrapped} layers")
print(f"Trainable: {trainable/1e6:.3f}M  /  Total: {total/1e6:.1f}M  ({100*trainable/total:.2f}%)")
assert n_wrapped == 36, (
    f"Expected 36 (12 blocks x (c_attn + attn.c_proj + mlp.c_proj)), got {n_wrapped}"
)
assert trainable < 1_500_000, "Way too many trainable params — check freeze_non_lora."
assert_same_device(model)

**Q1. (5 points)** What fraction of the original 124M parameters did you make trainable? With $r=8$ and target modules `(c_attn, c_proj)`, you should be around 0.6–0.7%. Think about why: how many params does each LoRA pair add for a `(d_in, d_out)` linear, and what are the dims of the three target layer types?

`# YOUR ANSWER:`

## Per-pair formula
- For a `(d_in, d_out)` linear, a single LoRA pair (A of shape `(r, d_in)`, B of shape `(d_out, r)`) adds #YOUR ANSWER HERE# parameters.

## Per-layer LoRA params at r=8
- `attn.c_attn`  (768 → 2304): #YOUR ANSWER HERE# params
- `attn.c_proj`  (768 →  768): #YOUR ANSWER HERE# params
- `mlp.c_proj`   (3072 → 768): #YOUR ANSWER HERE# params

## Totals
- Trainable parameters across all 12 blocks: #YOUR ANSWER HERE#
- Fraction of GPT-2 small (124M): #YOUR ANSWER HERE# %


## 6 · Dataset: TinyShakespeare

A single ~1MB text file. We tokenize the whole thing once, then chunk into fixed-length blocks.


In [ ]:
import urllib.request

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
TINY_SHAKES_URL = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
TINY_SHAKES_PATH = os.path.join(DATA_DIR, "tinyshakespeare.txt")
if not os.path.exists(TINY_SHAKES_PATH):
    urllib.request.urlretrieve(TINY_SHAKES_URL, TINY_SHAKES_PATH)

with open(TINY_SHAKES_PATH) as f:
    raw_text = f.read()

print(f"Corpus: {len(raw_text):,} chars")
print(raw_text[:300])

In [ ]:
BLOCK_SIZE = 256


class LMChunks(Dataset):
    """Tokenize a long string once, then yield contiguous fixed-length chunks for next-token LM training."""

    def __init__(self, text: str, tokenizer, block_size: int):
        self.block_size = block_size
        ids = tokenizer(text, return_tensors="pt").input_ids[0]
        # Drop the tail so we have a clean multiple of block_size.
        n_chunks = ids.size(0) // block_size
        self.ids = ids[: n_chunks * block_size].view(n_chunks, block_size)

    def __len__(self) -> int:
        return self.ids.size(0)

    def __getitem__(self, idx: int) -> dict:
        chunk = self.ids[idx]
        # For causal LM, labels = input_ids; HF will internally shift by one.
        return {"input_ids": chunk, "labels": chunk.clone()}


full_ds = LMChunks(raw_text, tokenizer, BLOCK_SIZE)
n_val = max(1, len(full_ds) // 20)
train_ds, val_ds = torch.utils.data.random_split(
    full_ds, [len(full_ds) - n_val, n_val], generator=torch.Generator().manual_seed(SEED)
)
print(f"Train chunks: {len(train_ds)} | Val chunks: {len(val_ds)} | block_size={BLOCK_SIZE}")

## 6.5 · Evaluation harness: perplexity

We need a quantitative sanity check, not just vibes from generated samples. Two metrics:

1. **Shakespeare-val PPL** — should drop sharply after fine-tuning (the model is getting better at predicting Shakespeare tokens).
2. **General-English PPL** — computed on a held-out non-Shakespeare snippet (Pride & Prejudice). This catches **catastrophic forgetting**: if it skyrockets, your LoRA has destroyed general English ability in chasing Shakespeare style. Mild rises are normal; large rises mean your `r`/`alpha`/lr is too aggressive or you're training too long.

> Note: right now `model` already has LoRA injected with $B = 0$, so its outputs are mathematically identical to plain GPT-2. The "baseline" PPL we print below is therefore the un-tuned GPT-2's PPL on each corpus.


In [ ]:
def collate(batch):
    return {k: torch.stack([b[k] for b in batch], dim=0) for k in batch[0]}


@torch.no_grad()
def compute_ppl(model: nn.Module, dataset, batch_size: int = 8) -> float:
    """Token-weighted perplexity over a chunk dataset. Lower = better."""
    model.eval()
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, collate_fn=collate)
    total_loss = 0.0
    total_tokens = 0
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        # HF computes mean cross-entropy over (B*T) shifted-label positions; weight by that token count.
        n_tok = batch["labels"].numel()
        total_loss += out.loss.item() * n_tok
        total_tokens += n_tok
    return math.exp(total_loss / total_tokens)

In [ ]:
# Forgetting-control corpus: a public-domain Pride & Prejudice excerpt (no Shakespeare in sight).
control_text = """It is a truth universally acknowledged, that a single man in possession of a good fortune, must be in want of a wife. However little known the feelings or views of such a man may be on his first entering a neighbourhood, this truth is so well fixed in the minds of the surrounding families, that he is considered as the rightful property of some one or other of their daughters.

"My dear Mr. Bennet," said his lady to him one day, "have you heard that Netherfield Park is let at last?"

Mr. Bennet replied that he had not.

"But it is," returned she; "for Mrs. Long has just been here, and she told me all about it."

Mr. Bennet made no answer.

"Do not you want to know who has taken it?" cried his wife impatiently.

"You want to tell me, and I have no objection to hearing it."

This was invitation enough.

"Why, my dear, you must know, Mrs. Long says that Netherfield is taken by a young man of large fortune from the north of England; that he came down on Monday in a chaise and four to see the place, and was so much delighted with it that he agreed with Mr. Morris immediately; that he is to take possession before Michaelmas, and some of his servants are to be in the house by the end of next week."

"What is his name?"

"Bingley."

"Is he married or single?"

"Oh! single, my dear, to be sure! A single man of large fortune; four or five thousand a year. What a fine thing for our girls!"

"How so? how can it affect them?"

"My dear Mr. Bennet," replied his wife, "how can you be so tiresome! You must know that I am thinking of his marrying one of them."

"Is that his design in settling here?"

"Design! nonsense, how can you talk so! But it is very likely that he may fall in love with one of them, and therefore you must visit him as soon as he comes."
"""

control_ds = LMChunks(control_text, tokenizer, BLOCK_SIZE)
print(f"Control chunks: {len(control_ds)} (block_size={BLOCK_SIZE})")
assert len(control_ds) >= 1, "Control corpus too short — add more text."

In [ ]:
ppl_shakes_before = compute_ppl(model, val_ds)
ppl_control_before = compute_ppl(model, control_ds)
print(f"Baseline Shakespeare-val PPL : {ppl_shakes_before:7.2f}")
print(f"Baseline Control      PPL    : {ppl_control_before:7.2f}")

## 7 · Training loop

The loop itself is given. **TODO 7.1**: build the optimizer over only the trainable parameters.


In [ ]:
@dataclass
class TrainCfg:
    epochs: int = 2
    batch_size: int = 8
    lr: float = 3e-4
    weight_decay: float = 0.0
    warmup_ratio: float = 0.05
    log_every: int = 50
    grad_clip: float = 1.0


def train_lora(model: nn.Module, train_ds, val_ds, cfg: TrainCfg) -> dict:
    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, collate_fn=collate)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate)

    # TODO 7.1: build an AdamW over only the parameters with requires_grad=True.
    # (Passing all params still trains correctly because frozen ones have no grads,
    # but it wastes optimizer-state memory — a real cost on bigger models.)
    optim = ...

    total_steps = len(train_loader) * cfg.epochs
    warmup_steps = int(cfg.warmup_ratio * total_steps)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(0.0, 0.5 * (1 + math.cos(math.pi * progress)))

    sched = torch.optim.lr_scheduler.LambdaLR(optim, lr_lambda)

    history = {"train_loss": [], "val_loss": []}
    step = 0
    for epoch in range(cfg.epochs):
        model.train()
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            out = model(**batch)
            loss = out.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], cfg.grad_clip)
            optim.step()
            sched.step()
            optim.zero_grad(set_to_none=True)

            if step % cfg.log_every == 0:
                history["train_loss"].append((step, loss.item()))
                print(f"epoch {epoch} step {step:4d} | train loss {loss.item():.4f} | lr {sched.get_last_lr()[0]:.2e}")
            step += 1

        # validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                val_losses.append(model(**batch).loss.item())
        v = sum(val_losses) / len(val_losses)
        history["val_loss"].append((step, v))
        print(f"== epoch {epoch} val loss {v:.4f} (ppl {math.exp(v):.2f}) ==")

    return history


cfg = TrainCfg(epochs=2, batch_size=8, lr=3e-4)
history = train_lora(model, train_ds, val_ds, cfg)

## 8 · Generate after fine-tuning

Same prompts, same seed. Read the outputs and compare to the baseline.


In [ ]:
print("=== AFTER LoRA FINE-TUNE ===")
for p in PROMPTS:
    print("-" * 60)
    print(generate(p, seed=SEED))


In [ ]:
ppl_shakes_after = compute_ppl(model, val_ds)
ppl_control_after = compute_ppl(model, control_ds)
print(f"{'metric':<30}{'before':>10}{'after':>10}{'delta':>10}")
print(f"{'Shakespeare-val PPL':<30}{ppl_shakes_before:>10.2f}{ppl_shakes_after:>10.2f}{ppl_shakes_after - ppl_shakes_before:>+10.2f}")
print(f"{'Control (P&P) PPL':<30}{ppl_control_before:>10.2f}{ppl_control_after:>10.2f}{ppl_control_after - ppl_control_before:>+10.2f}")

**Q2. (5 points)** Qualitatively, how did the outputs change? Mention at least two specific shifts you can point to (vocabulary, punctuation/structure, character names, line breaks, etc.).

`# YOUR ANSWER:`

## Stylistic shifts after fine-tuning
- Shift 1 (vocabulary / diction): #YOUR ANSWER HERE#
- Shift 2 (formatting / structure): #YOUR ANSWER HERE#
- Shift 3 (optional, e.g. named entities): #YOUR ANSWER HERE#

**Q3. (5 points)** Report the four PPL numbers above. Did the control PPL move? In which direction, and by how much relative to the Shakespeare drop? What does this tell you about the trade-off you're making with LoRA fine-tuning?

`# YOUR ANSWER:`

## PPL numbers
| metric              | before  | after   |
|---------------------|--------:|--------:|
| Shakespeare-val PPL | #YOUR ANSWER HERE# | #YOUR ANSWER HERE# |
| Control (P&P) PPL   | #YOUR ANSWER HERE# | #YOUR ANSWER HERE# |

## Direction and magnitude
- Control PPL moved (up / down / unchanged): #YOUR ANSWER HERE#
- Magnitude relative to the Shakespeare-PPL drop (smaller / similar / larger): #YOUR ANSWER HERE#

## Trade-off interpretation
   #YOUR ANSWER HERE#


## 9 · TODO: Save and load just the adapter (5 points)

A LoRA adapter is tiny (megabytes). Below, save **only** the LoRA parameters to disk, then verify you can load them back into a fresh GPT-2 and reproduce the trained model's outputs.


In [ ]:
ADAPTER_PATH = "lora_adapter.pt"


def save_adapter(model: nn.Module, path: str) -> None:
    # TODO 9.1: collect a state_dict containing only parameters whose name contains "lora_".
    # Save it with torch.save.
    ...


def load_adapter(model: nn.Module, path: str) -> None:
    # TODO 9.2: torch.load and copy the saved tensors into the matching parameters of model.
    # Use strict matching: every saved key must exist in model.
    ...


save_adapter(model, ADAPTER_PATH)
print(f"Adapter file size: {os.path.getsize(ADAPTER_PATH) / 1e6:.2f} MB")

In [ ]:
# Round-trip check: build a fresh GPT-2 + LoRA, load the adapter, confirm outputs match.
fresh = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)
replace_conv1d_with_linear(fresh)
inject_lora(fresh, target_names=("c_attn", "c_proj"), r=8, alpha=16, dropout=0.05)
freeze_non_lora(fresh)
load_adapter(fresh, ADAPTER_PATH)
assert_same_device(fresh)

# IMPORTANT: a freshly-constructed nn.Module defaults to train mode, which leaves
# both GPT-2's own dropouts (attn_pdrop, resid_pdrop) and LoRALinear's dropout
# active. torch.no_grad() turns off autograd but NOT dropout — that's controlled
# by module.training. So set both models to eval before comparing logits.
model.eval()
fresh.eval()

with torch.no_grad():
    ids = tokenizer("ROMEO:", return_tensors="pt").input_ids.to(device)
    a = model(ids).logits
    b = fresh(ids).logits

print("max abs diff:", (a - b).abs().max().item())
assert torch.allclose(a, b, atol=1e-5), "Adapter round-trip failed."
print("OK: adapter round-trips.")

## 10 · TODO: Compare with the `peft` library (5 points)

Re-run the same fine-tune using `peft.LoraConfig` + `get_peft_model` on a fresh GPT-2 (with the original `Conv1D` layers — `peft` knows how to handle them). Match the hyperparameters: `r=8`, `alpha=16`, `dropout=0.05`, target modules `c_attn` and `c_proj`.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

peft_base = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

# TODO 10.1: build a LoraConfig matching your hand-rolled setup.
peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=...,
    lora_alpha=...,
    lora_dropout=...,
    target_modules=...,  # Hint: a list of module-name substrings. ["c_attn", "c_proj"] works for GPT-2.
    bias="none",
)
peft_model = get_peft_model(peft_base, peft_config).to(device)  # belt-and-suspenders: ensure adapter weights land on GPU
peft_model.print_trainable_parameters()
assert_same_device(peft_model)

In [ ]:
# TODO 10.2: train peft_model with the same TrainCfg you used above.
# Tip: peft_model behaves like a normal nn.Module; the train_lora() function works as-is.
peft_history = ...

In [ ]:
# Generate from the PEFT-trained model with the same prompts/seed.
print("=== PEFT-trained ===")
backup = model
model = peft_model  # so generate() uses it
for p in PROMPTS:
    print("-" * 60)
    print(generate(p, seed=SEED))
model = backup

In [ ]:
# Quantitative comparison: PEFT vs hand-rolled, on the same val/control sets.
ppl_shakes_peft = compute_ppl(peft_model, val_ds)
ppl_control_peft = compute_ppl(peft_model, control_ds)
print(f"{'metric':<30}{'hand-rolled':>14}{'PEFT':>10}")
print(f"{'Shakespeare-val PPL':<30}{ppl_shakes_after:>14.2f}{ppl_shakes_peft:>10.2f}")
print(f"{'Control (P&P) PPL':<30}{ppl_control_after:>14.2f}{ppl_control_peft:>10.2f}")

**Q4.(5 points)** Compare your hand-rolled trainable parameter count vs `peft_model.print_trainable_parameters()`. Are they identical? If not, where does the difference come from? (Hint: think about which projections `peft` decides to wrap given `target_modules=["c_attn", "c_proj"]`.)

`# YOUR ANSWER:`

## Param counts
- Hand-rolled trainable params: #YOUR ANSWER HERE#
- PEFT trainable params: #YOUR ANSWER HERE#
- Identical? (yes / no): #YOUR ANSWER HERE#

## Where any difference comes from (or why they match):
 #YOUR ANSWER HERE#

**Q5.(5 points)** After the same number of training steps, did your hand-rolled LoRA reach a similar Shakespeare-val PPL to the PEFT version? List one reason they could differ even with matched hyperparameters.

`# YOUR ANSWER:`

## Shakespeare-val PPL comparison
- Hand-rolled: #YOUR ANSWER HERE#
- PEFT: #YOUR ANSWER HERE#
- Within ~10% of each other? (yes / no): #YOUR ANSWER HERE#

## One reason they could still differ with matched hyperparameters:
 #YOUR ANSWER HERE#


## 11 · Bonus: pick a different style

Pick one (or all three!) and watch the model become a different character. Each option below builds a `raw_text` string; everything downstream of `LMChunks(raw_text, …)` is identical.

> **Tip:** for any of these, **reset the model** before the new fine-tune (re-run cell 5 first), or you'll be training a Shakespeare-flavoured Rick. Or do that on purpose. It's funny.


### 11A · Rick & Morty

A community dialogue dataset is hosted on the Hugging Face Hub. Substring filter to Rick's lines for stronger style transfer.


In [ ]:
# Bonus A: Rick & Morty dialogue.
# This uses the `Prarabdha/Rick_and_Morty_Transcript` dataset on HF Hub.
# If that dataset is unavailable, swap in any other dialogue source — what matters is the format.
from datasets import load_dataset

try:
    ds = load_dataset("Prarabdha/Rick_and_Morty_Transcript", split="train")
    # Inspect: print(ds.column_names, ds[0])
    # Filter to Rick's lines and concatenate.
    rick_lines = [row["dialouge"] for row in ds if row.get("speaker", "").strip().lower() == "rick"]
    raw_text_rick = "\n".join(f"Rick: {line}" for line in rick_lines if isinstance(line, str))
    print(f"Rick corpus: {len(raw_text_rick):,} chars over {len(rick_lines)} lines")
    print(raw_text_rick[:400])
except Exception as e:
    print("Dataset load failed:", e)
    print("Fallback: paste your own transcript text into raw_text_rick.")
    raw_text_rick = ""

# To use this corpus, set `raw_text = raw_text_rick` and re-run the dataset / training cells.

### 11B · Yoda

Yoda has a small but iconic corpus. We ship ~50 quotes inline as a starter — augment with your own scraping for stronger transfer (e.g., parse Star Wars scripts on imsdb.com).


In [ ]:
# Bonus B: Yoda. Inline starter corpus — feel free to extend.
yoda_quotes = [
    "Do or do not. There is no try.",
    "Fear is the path to the dark side. Fear leads to anger. Anger leads to hate. Hate leads to suffering.",
    "Size matters not. Look at me. Judge me by my size, do you?",
    "When nine hundred years old you reach, look as good you will not.",
    "Wars not make one great.",
    "Adventure. Heh. Excitement. Heh. A Jedi craves not these things.",
    "Patience you must have, my young padawan.",
    "Truly wonderful, the mind of a child is.",
    "Always pass on what you have learned.",
    "Train yourself to let go of everything you fear to lose.",
    "Difficult to see. Always in motion is the future.",
    "The greatest teacher, failure is.",
    "Pass on what you have learned. Strength. Mastery.",
    "Named must your fear be before banish it you can.",
    "Powerful you have become, the dark side I sense in you.",
    "You must unlearn what you have learned.",
    "That is why you fail.",
    "Once you start down the dark path, forever will it dominate your destiny.",
    "Mind what you have learned. Save you it can.",
    "Luminous beings are we, not this crude matter.",
    "Through the Force, things you will see. Other places. The future, the past. Old friends long gone.",
    "Decide you must, how to serve them best.",
    "Hard to see, the dark side is.",
    "Strong with the Force, young Skywalker is.",
    "Many of the truths that we cling to depend on our point of view.",
    "Smaller in number are we, but larger in mind.",
    "Around the survivors a perimeter create.",
    "Rejoice for those around you who transform into the Force.",
    "Mourn them, do not. Miss them, do not.",
    "When you look at the dark side, careful you must be.",
    "Help you I can. Yes. Mmm.",
    "Already know you that which you need.",
    "A Jedi's strength flows from the Force.",
    "Anger, fear, aggression. The dark side are they.",
    "If once you start down the dark path, forever will it dominate your destiny.",
    "Consume you it will, as it did Obi-Wan's apprentice.",
    "Do not underestimate the powers of the Emperor or suffer your father's fate, you will.",
    "Reckless he is. Matters are worse.",
    "Control, control. You must learn control.",
    "Yes, a Jedi's strength flows from the Force. But beware. Anger, fear, aggression.",
    "Stopped they must be. On this all depends.",
    "Faith in your friends, yours is.",
    "Soon will I rest. Yes, forever sleep. Earned it I have.",
    "Twilight is upon me, and soon night must fall.",
    "When gone am I, the last of the Jedi will you be.",
    "The Force runs strong in your family. Pass on what you have learned.",
    "There is another. Sky-walker.",
    "No. Try not. Do, or do not. There is no try.",
    "Judge me by my size, do you?",
    "Concentrate. Feel the Force flow.",
]

# Repeat to give the model enough tokens to chunk into BLOCK_SIZE windows.
raw_text_yoda = ("\n".join(yoda_quotes) + "\n") * 200
print(f"Yoda corpus: {len(raw_text_yoda):,} chars (after replication)")
print(raw_text_yoda[:300])


**Q6 (bonus).** Pick at least one of A/B/C, fine-tune, and paste 3 generated samples that show the style. Also report Shakespeare-val PPL and control PPL after this run — does Shakespeare PPL get worse (it should: you're tuning toward a different style now)? One sentence on what surprised you most.

`# YOUR ANSWER:`

## Bonus run setup
- Dataset chosen (A=Rick & Morty, B=Hermione, C=Yoda): #YOUR ANSWER HERE#

## Three generated samples that show the style
1. #YOUR ANSWER HERE#
2. #YOUR ANSWER HERE#
3. #YOUR ANSWER HERE#

## PPL after the bonus fine-tune
- Shakespeare-val PPL: #YOUR ANSWER HERE#
- Control (P&P) PPL: #YOUR ANSWER HERE#
- Did Shakespeare PPL get worse vs the previous run? (yes / no): #YOUR ANSWER HERE#

## Most surprising observation (one sentence)
 #YOUR ANSWER HERE#


## 12 · Submission checklist

- [ ] All cells run top to bottom without error.
- [ ] Q1–Q5 answered. Q6 if you did the bonus.
- [ ] Baseline and post-fine-tune sample outputs are visible in the saved notebook.
- [ ] All four PPL numbers (Shakespeare/control × before/after) are printed.
- [ ] `lora_adapter.pt` round-trip assertion passes.
- [ ] Hand-rolled vs PEFT comparison runs and PPLs are reported side-by-side.

Good luck — and may the gradients flow strong with you.
